This is a **CLEAN** but vivid version of codes (along with visualizations) 
for analyzing the experiment ***freight collaboration-chessboard*** results

In [ ]:
from dataclasses import dataclass
from enum import Enum
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns
# Use repo-relative path so it works on other machines
notebook_dir = Path.cwd() / "python" / "test"
if notebook_dir.exists() and str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
import matsim
import matsim_output_reader
import metric_anls
# --- reload the module to reflect any changes made during development ---
import importlib
importlib.reload(matsim_output_reader)
importlib.reload(metric_anls)

# Configuration

In [ ]:
# Resolve analysis path relative to repo root
repo_root = notebook_dir.parents[1]  # .../matsim-libs-2024
anls_path = repo_root / "output" / "chessboardCarrierReceiverCollab"

class DepotLocation(Enum):
    INSIDE = 'center'
    OUTSIDE = 'left'

class ReceiverDistribution(Enum):
    DISPERSED = 'DISPERSED'
    CLUSTERED = 'CLUSTERED'
    RANDOM = 'FULLY_RANDOM'

ALLOCATION_FACTOR = 0.8
#--- Penalty ---#
PENALTY_LIST = [0, 0.0003, 0.0008, 0.0014, 0.0028, 0.0056,
                0.0098, 0.014, 0.0167, 0.0222, 0.028]
PENALTY_LIST_SCALE = [round(x * 3600) for x in PENALTY_LIST]  # scale to avoid float precision issues
PENALTY_LIST_SCALE[-1] = 100 
PENALTY_DICT = {k: v for k, v in zip(PENALTY_LIST, PENALTY_LIST_SCALE)}
#--- Penalty ---#

TOTAL_INSTANCES = 50


In [ ]:
#--- Generate keywords for each scenario ---#
# Example tuple: ("center", "dispersed", 0)
scenario_keywords = [
    (depot, receiver, penalty)
    for depot, receiver, penalty in product([DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value],
                                            [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value], 
                                            PENALTY_LIST)
]
scenario_keywords[:3]

# Test

In [ ]:
test_folder = os.listdir(anls_path)[5]
print(f"Analyzing results from folder: {test_folder}")
test_carriers_df, test_shipments_df = matsim_output_reader.read_carriers(
    os.path.join(anls_path, test_folder, 'output_carriers.xml.gz')
)
test_receivers_dict = matsim_output_reader.read_receivers(
    os.path.join(anls_path, test_folder, 'receivers.xml.gz'), (6,8))

test_iter0_carrier_df, test_iter0_shipment_df = matsim_output_reader.read_iter0_carriers(
    os.path.join(anls_path, test_folder), False)

test_iter0_carrier_score_dict = matsim_output_reader.read_iter0_carriers(
    os.path.join(anls_path, test_folder))

In [ ]:
matsim_network = matsim.read_network(os.path.join(anls_path, test_folder, 'output_network.xml.gz'))
network_link_df = matsim_network.links
network_node_df = matsim_network.nodes

In [ ]:
network_link_df

## Test for reader

In [ ]:
test_carriers_df

In [ ]:
test_shipments_df

In [ ]:
test_iter0_carrier_df

In [ ]:
test_iter0_shipment_df

In [ ]:
test_iter0_carrier_score_dict

In [ ]:
test_receivers_dict

In [ ]:
test_collaboration_data = matsim_output_reader.read_collaboration_allocation_data(os.path.join(anls_path, test_folder,))
test_collaboration_data

## Test for metric analysis

In [ ]:
test_fv_travel_chains = metric_anls.derive_freight_vehicle_travel_chains(
    os.path.join(anls_path, test_folder, 'output_events.xml.gz')
)
test_fv_travel_chains

In [ ]:
test_iter0_fv_travel_chains = metric_anls.derive_freight_vehicle_travel_chains(
    os.path.join(anls_path, test_folder, 'ITERS/it.0/0.events.xml.gz')
)
test_iter0_fv_travel_chains

In [ ]:
test_freight_vkt = metric_anls.compute_freight_vehicle_km_traveled(
    test_fv_travel_chains,
    network_link_df,
)
test_freight_vkt

In [ ]:
test_iter0_fv_vkt = metric_anls.compute_freight_vehicle_km_traveled(
    test_iter0_fv_travel_chains,
    network_link_df,
)
test_iter0_fv_vkt

In [ ]:
test_fv_travel_times = metric_anls.compute_freight_vehicle_travel_times(
    os.path.join(anls_path, test_folder, 'output_events.xml.gz')
)
test_fv_travel_times

In [ ]:
test_iter0_fv_travel_times = metric_anls.compute_freight_vehicle_travel_times(
    os.path.join(anls_path, test_folder, 'ITERS/it.0/0.events.xml.gz')
)
test_iter0_fv_travel_times

In [ ]:
test_shipment_travel_timeAndDist = metric_anls.compute_freight_shipment_travel_distance_and_time(
    os.path.join(anls_path, test_folder, 'output_events.xml.gz'),
    network_link_df
)
test_shipment_travel_timeAndDist

In [ ]:
print("Total travel time (seconds):", test_shipment_travel_timeAndDist['travel_time_seconds'].sum())
print("Total shipment travel distance (kilometers):", test_shipment_travel_timeAndDist['travel_distance_km'].sum())

In [ ]:
test_iter0_shipment_travel_timeAndDist = metric_anls.compute_freight_shipment_travel_distance_and_time(
    os.path.join(anls_path, test_folder, 'ITERS/it.0/0.events.xml.gz'),
    network_link_df
)
test_iter0_shipment_travel_timeAndDist

In [ ]:
print("Total travel time (seconds):", test_iter0_shipment_travel_timeAndDist['travel_time_seconds'].sum())
print("Total shipment travel distance (kilometers):", test_iter0_shipment_travel_timeAndDist['travel_distance_km'].sum())   


In [ ]:
test_missed_tw = metric_anls.analyse_missed_time_windows(
    os.path.join(anls_path, test_folder, 'output_events.xml.gz'),
    test_shipments_df,
)
test_missed_tw

In [ ]:
test_iter0_missed_tw = metric_anls.analyse_missed_time_windows(
    os.path.join(anls_path, test_folder, 'ITERS/it.0/0.events.xml.gz'),
    test_iter0_shipment_df,
)
test_iter0_missed_tw

# Aggregate Analysis

Use `agg_anls` module to process all scenarios and generate outputs.

In [ ]:
# Import the aggregate analysis module
import agg_anls
importlib.reload(agg_anls)

# Configuration for batch processing
INPUT_PATH = str(anls_path)
OUTPUT_PATH = str(repo_root / "data" / "freightChessboardRC" / "clean")

# Define which scenarios to process
DEPOT_LOCATIONS = [DepotLocation.INSIDE.value, DepotLocation.OUTSIDE.value]
RECEIVER_DISTRIBUTIONS = [ReceiverDistribution.DISPERSED.value, ReceiverDistribution.CLUSTERED.value]
ORIGINAL_TW = (6, 8)  # Original time window in hours
LAST_ITER = 50  # Last iteration number

print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Depot locations: {DEPOT_LOCATIONS}")
print(f"Receiver distributions: {RECEIVER_DISTRIBUTIONS}")
print(f"Penalties: {PENALTY_LIST}")

## Test Single Scenario Processing

Before running batch processing, test with a single scenario to verify the functions work correctly.

In [ ]:
# Test parsing folder name
test_folder_name = test_folder
test_config = agg_anls.parse_folder_name(test_folder_name, str(anls_path))
print(f"Parsed config: {test_config}")

In [ ]:
# Build network graph once for efficiency
network_graph = agg_anls.build_network_graph(network_link_df, network_node_df)
print(f"Network graph: {network_graph.number_of_nodes()} nodes, {network_graph.number_of_edges()} edges")

In [ ]:
# Test compute_scenario_metrics for single scenario
if test_config:
    test_metrics = agg_anls.compute_scenario_metrics(
        test_config,
        network_link_df,
        network_node_df,
        network_graph,
        original_tw=ORIGINAL_TW,
        last_iter=LAST_ITER,
        compute_network_distances=True
    )
# Display as DataFrame for better visualization
pd.DataFrame([test_metrics])

In [ ]:
# Test compute_merged_shipment_df
if test_config:
    test_merged_shipment_df = agg_anls.compute_merged_shipment_df(test_config, network_link_df)
    print(f"Merged shipment DataFrame shape: {test_merged_shipment_df.shape}")
test_merged_shipment_df

In [ ]:
# Test create_geo_dataframe
if test_config:
    test_geo_df = agg_anls.create_geo_dataframe(
        test_config,
        network_node_df,
        network_link_df,
        original_tw=ORIGINAL_TW,
        last_iter=LAST_ITER
    )
    print(f"GeoDataFrame shape: {test_geo_df.shape}")
test_geo_df

## Batch Processing

Run the batch processing for all scenarios. This will:
1. Process each matching scenario folder
2. Compute metrics and save to `metrics.csv.gz`
3. Create merged shipment data and save to `shipments.csv.gz`
4. Create GeoDataFrame and save to `geo_data.geojson`
5. Save combined metrics to `all_scenarios_metrics.csv.gz`

In [ ]:
# Option 1: Use quick_analyze with default parameters
# all_metrics_df = agg_anls.quick_analyze(
#     input_path=INPUT_PATH,
#     output_path=OUTPUT_PATH,
#     depot_locations=DEPOT_LOCATIONS,
#     receiver_distributions=RECEIVER_DISTRIBUTIONS,
#     penalty_list=PENALTY_LIST,
#     total_instances=TOTAL_INSTANCES,
#     original_tw=ORIGINAL_TW,
#     compute_network_distances=True
# )

# Option 2: Use process_all_scenarios with custom parameters (for testing with fewer scenarios)
# Test with just a subset first
test_all_metrics_df = agg_anls.process_all_scenarios(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    depot_locations=['left'],  # Test with one depot location
    receiver_distributions=['CLUSTERED'],  # Test with one distribution
    penalties=[0.0],  # Test with one penalty
    instances=[0],  # Test with first 2 instances only
    original_tw=ORIGINAL_TW,
    last_iter=LAST_ITER,
    compute_network_distances=True,
    verbose=True
)

In [ ]:
# View the test results
test_all_metrics_df

# Analyse agg_file

## Read agg_metric df

In [ ]:
all_metrics_df = pd.read_csv(os.path.join(OUTPUT_PATH, 'all_scenarios_metrics.csv.gz'), compression='gzip')
all_metrics_df

In [ ]:
all_metrics_df.query("depot_location == 'center' and receiver_distribution == 'CLUSTERED' and instance == 41")

In [ ]:
all_metrics_df['collaboration_rate'] = all_metrics_df['num_collaborative_receivers'] / 10
all_metrics_df

In [ ]:
all_metrics_random_df = all_metrics_df[all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value]
all_metrics_random_df

In [ ]:
all_metrics_center_random_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value)]
all_metrics_center_random_df

In [ ]:
all_metrics_outside_random_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.RANDOM.value)]
all_metrics_outside_random_df

In [ ]:
all_metrics_center_clustered_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)]
all_metrics_center_dispersed_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.INSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)]
all_metrics_outside_clustered_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.CLUSTERED.value)]
all_metrics_outside_dispersed_df = all_metrics_df[(all_metrics_df['depot_location'] == DepotLocation.OUTSIDE.value) & (all_metrics_df['receiver_distribution'] == ReceiverDistribution.DISPERSED.value)]

## Analyse random scenario before-and-after operation

In [ ]:
def agg_random_df_across_instances(agg_metrics_df: pd.DataFrame,
                                   col_split: str = 'instance') -> pd.DataFrame:
    """
    Aggregate metrics DataFrame across instances by averaging metric values.
    """
    index_col_split = agg_metrics_df.columns.to_list().index(col_split)
    value_cols = agg_metrics_df.columns.tolist()[index_col_split + 1:]
    agg_df = agg_metrics_df.pivot_table(
        index=['depot_location', 'receiver_distribution', 'allocation_factor', 'penalty'],
        values=value_cols,
        aggfunc='mean'
    ).reset_index()
    return agg_df

In [ ]:
def boxplot_metric(all_metric_df: pd.DataFrame, 
                   metric: str, 
                   title: str,
                   iter0_metric=None,
                   compare_iter0: bool = False):
    """
    Raincloud plot of a given metric across penalties.
    If iter0_metric is provided or compare_iter0 is True, plot 2 clouds per penalty for comparison.
    """
    main_color = '#86c5da'
    secondary_color = '#90cd97'

    # Ensure consistent penalty order
    penalty_order = sorted(all_metric_df["penalty"].unique())
    n_penalties = len(penalty_order)

    if compare_iter0 or iter0_metric is not None:
        # Resolve iter0 values from input
        if iter0_metric is None:
            candidate_cols = [
                f"iter0_{metric}",
                f"{metric}_iter0",
                f"{metric}_it0",
            ]
            iter0_col = next((c for c in candidate_cols if c in all_metric_df.columns), None)
            if iter0_col is None:
                raise ValueError(
                    "compare_iter0=True but no iter0 metric column found. "
                    "Provide iter0_metric or add iter0 column."
                )
            iter0_values = all_metric_df[iter0_col]
        elif isinstance(iter0_metric, str):
            if iter0_metric not in all_metric_df.columns:
                raise ValueError(f"iter0_metric column '{iter0_metric}' not found in DataFrame.")
            iter0_values = all_metric_df[iter0_metric]
        elif isinstance(iter0_metric, pd.DataFrame):
            if metric in iter0_metric.columns:
                iter0_values = iter0_metric[metric]
            else:
                iter0_values = iter0_metric.iloc[:, 0]
            iter0_values = iter0_values.reindex(all_metric_df.index)
        else:
            iter0_values = pd.Series(iter0_metric, index=all_metric_df.index)

        plot_df = pd.concat(
            [
                pd.DataFrame({
                    "penalty": all_metric_df["penalty"],
                    "value": all_metric_df[metric],
                    "group": "last_iter",
                }),
                pd.DataFrame({
                    "penalty": all_metric_df["penalty"],
                    "value": iter0_values,
                    "group": "iter0",
                }),
            ],
            ignore_index=True,
        )

        fig, ax = plt.subplots(figsize=(max(8, n_penalties * 0.8), 5))
        
        # Map penalty to numeric positions
        penalty_to_pos = {p: i for i, p in enumerate(penalty_order)}
        plot_df['x_pos'] = plot_df['penalty'].map(penalty_to_pos)
        
        offset = 0.2  # offset between groups
        
        for i, (grp, color) in enumerate([('last_iter', main_color), ('iter0', secondary_color)]):
            grp_df = plot_df[plot_df['group'] == grp]
            positions = grp_df['x_pos'].unique()
            shift = -offset if i == 0 else offset
            
            for pos in positions:
                data = grp_df[grp_df['x_pos'] == pos]['value'].dropna().values
                if len(data) == 0:
                    continue
                x = pos + shift
                
                # Half-violin (cloud) - only right side for last_iter, left side for iter0
                vp = ax.violinplot([data], positions=[x], widths=0.35, showextrema=False)
                for body in vp['bodies']:
                    m = body.get_paths()[0].vertices[:, 0].mean()
                    if i == 0:  # last_iter: show right half
                        body.get_paths()[0].vertices[:, 0] = np.clip(
                            body.get_paths()[0].vertices[:, 0], m, None)
                    else:  # iter0: show left half
                        body.get_paths()[0].vertices[:, 0] = np.clip(
                            body.get_paths()[0].vertices[:, 0], None, m)
                    body.set_facecolor(color)
                    body.set_edgecolor('black')
                    body.set_linewidth(0.8)
                    body.set_alpha(0.7)
                
                # Boxplot (small, horizontal style)
                bp = ax.boxplot([data], positions=[x], widths=0.12, vert=True, patch_artist=True,
                                showfliers=False, manage_ticks=False)
                bp['boxes'][0].set_facecolor('white')
                bp['boxes'][0].set_edgecolor('black')
                bp['boxes'][0].set_linewidth(0.8)
                bp['medians'][0].set_color('black')
                bp['medians'][0].set_linewidth(1.2)
                for whisker in bp['whiskers']:
                    whisker.set_linewidth(0.8)
                for cap in bp['caps']:
                    cap.set_linewidth(0.8)
                
                # Strip (rain) - jittered points below
                jitter = np.random.uniform(-0.06, 0.06, len(data))
                ax.scatter(x + jitter, data, s=8, alpha=0.4, c='black', zorder=3)
        
        ax.set_xticks(range(n_penalties))
        ax.set_xticklabels([str(p) for p in penalty_order], rotation=45, ha='right')
        
        # Legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=main_color, edgecolor='black', label='last_iter'),
                           Patch(facecolor=secondary_color, edgecolor='black', label='iter0')]
        ax.legend(handles=legend_elements, loc='upper right')
        
    else:
        fig, ax = plt.subplots(figsize=(max(8, n_penalties * 0.7), 5))
        
        penalty_to_pos = {p: i for i, p in enumerate(penalty_order)}
        all_metric_df = all_metric_df.copy()
        all_metric_df['x_pos'] = all_metric_df['penalty'].map(penalty_to_pos)
        
        for pos in range(n_penalties):
            data = all_metric_df[all_metric_df['x_pos'] == pos][metric].dropna().values
            if len(data) == 0:
                continue
            
            # Half-violin (cloud) - show right half only
            vp = ax.violinplot([data], positions=[pos], widths=0.5, showextrema=False)
            for body in vp['bodies']:
                m = body.get_paths()[0].vertices[:, 0].mean()
                body.get_paths()[0].vertices[:, 0] = np.clip(
                    body.get_paths()[0].vertices[:, 0], m, None)
                body.set_facecolor(main_color)
                body.set_edgecolor('black')
                body.set_linewidth(0.8)
                body.set_alpha(0.7)
            
            # Boxplot
            bp = ax.boxplot([data], positions=[pos], widths=0.15, vert=True, patch_artist=True,
                            showfliers=False, manage_ticks=False)
            bp['boxes'][0].set_facecolor('white')
            bp['boxes'][0].set_edgecolor('black')
            bp['boxes'][0].set_linewidth(0.8)
            bp['medians'][0].set_color('black')
            bp['medians'][0].set_linewidth(1.2)
            for whisker in bp['whiskers']:
                whisker.set_linewidth(0.8)
            for cap in bp['caps']:
                cap.set_linewidth(0.8)
            
            # Strip (rain) - jittered points on left side
            jitter = np.random.uniform(-0.18, -0.08, len(data))
            ax.scatter(pos + jitter, data, s=8, alpha=0.4, c='black', zorder=3)
        
        ax.set_xticks(range(n_penalties))
        ax.set_xticklabels([str(p) for p in penalty_order], rotation=45, ha='right')

    ax.set_xlabel("Penalty")
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
def scatter_3d(all_metrics_df: pd.DataFrame,
               x_col: str,
               y_col: str,
               z_col: str,
               title: str,
               color_col: str = None,
               figsize: tuple = (10, 8)):
    """
    Create a 3D scatter plot.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        z_col: Column name for z-axis
        title: Plot title
        color_col: Optional column for color encoding points
        figsize: Figure size tuple (width, height)
    """
    from mpl_toolkits.mplot3d import Axes3D
    
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    
    x = all_metrics_df[x_col]
    y = all_metrics_df[y_col]
    z = all_metrics_df[z_col]
    
    if color_col and color_col in all_metrics_df.columns:
        c = all_metrics_df[color_col]
        scatter = ax.scatter(x, y, z, c=c, cmap='viridis', alpha=0.7, edgecolors='k', linewidth=0.5)
        cbar = fig.colorbar(scatter, ax=ax, shrink=0.6, pad=0.1)
        cbar.set_label(color_col)
    else:
        ax.scatter(x, y, z, c='skyblue', alpha=0.7, edgecolors='k', linewidth=0.5)
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_zlabel(z_col)
    ax.set_title(title)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def scatter_color_encoded(all_metrics_df: pd.DataFrame,
                          x_col: str,
                          y_col: str,
                          z_color_col: str,
                          title: str,
                          cmap: str = 'viridis',
                          figsize: tuple = (8, 6),
                          show_colorbar: bool = True,
                          alpha: float = 0.7):
    """
    Plot 2D scatter with z-dimension encoded as color.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        z_color_col: Column name for color encoding (z-dimension)
        title: Plot title
        cmap: Colormap name (default: 'viridis')
        figsize: Figure size tuple
        show_colorbar: Whether to display colorbar
        alpha: Transparency of points
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    x = all_metrics_df[x_col]
    y = all_metrics_df[y_col]
    c = all_metrics_df[z_color_col]
    
    scatter = ax.scatter(x, y, c=c, cmap=cmap, alpha=alpha, 
                         edgecolors='k', linewidth=0.5, s=60)
    
    if show_colorbar:
        cbar = fig.colorbar(scatter, ax=ax)
        cbar.set_label(z_color_col)
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_bubble_chart(all_metrics_df: pd.DataFrame,
                      x_col: str,
                      y_col: str,
                      z_size_col: str,
                      title: str,
                      color_col: str = None,
                      size_scale: float = 200,
                      figsize: tuple = (8, 6),
                      alpha: float = 0.6):
    """
    Create a bubble chart with z-dimension represented by bubble size.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        z_size_col: Column name for bubble size (z-dimension)
        title: Plot title
        color_col: Optional column for color encoding
        size_scale: Scaling factor for bubble sizes
        figsize: Figure size tuple
        alpha: Transparency of bubbles
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    x = all_metrics_df[x_col]
    y = all_metrics_df[y_col]
    
    # Normalize size values to reasonable bubble sizes
    z_raw = all_metrics_df[z_size_col]
    z_min, z_max = z_raw.min(), z_raw.max()
    if z_max > z_min:
        sizes = ((z_raw - z_min) / (z_max - z_min) + 0.1) * size_scale
    else:
        sizes = size_scale * 0.5
    
    if color_col and color_col in all_metrics_df.columns:
        c = all_metrics_df[color_col]
        scatter = ax.scatter(x, y, s=sizes, c=c, cmap='viridis', alpha=alpha,
                             edgecolors='k', linewidth=0.5)
        cbar = fig.colorbar(scatter, ax=ax)
        cbar.set_label(color_col)
    else:
        ax.scatter(x, y, s=sizes, c='skyblue', alpha=alpha,
                   edgecolors='k', linewidth=0.5)
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    # Add size legend
    size_labels = [z_min, (z_min + z_max) / 2, z_max]
    size_handles = []
    for val in size_labels:
        if z_max > z_min:
            s = ((val - z_min) / (z_max - z_min) + 0.1) * size_scale
        else:
            s = size_scale * 0.5
        size_handles.append(ax.scatter([], [], s=s, c='gray', alpha=0.6, edgecolors='k'))
    
    legend_labels = [f'{val:.2g}' for val in size_labels]
    ax.legend(size_handles, legend_labels, title=z_size_col, 
              loc='upper right', framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def group_scatter_plot(all_metrics_df: pd.DataFrame,
                       x_col: str,
                       y_col: str,
                       z_col: str,
                       group_of_z_col: list,  # lists in list, like [[0,1,2], [3,4,5], [6,7,8]]
                       scheme: str = 'color',  # 'bubble' or 'color' strategy
                       title: str = '',
                       group_labels: list = None,
                       figsize: tuple = (8, 6),
                       alpha: float = 0.7,
                       size_scale: float = 200):
    """
    Create a scatter plot with z-dimension grouped into categories.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        z_col: Column name for grouping (z-dimension)
        group_of_z_col: List of lists defining value groups, e.g., [[0,1], [2,3], [4,5]]
        scheme: 'color' for color-encoded groups, 'bubble' for size-encoded groups
        title: Plot title
        group_labels: Optional list of labels for each group
        figsize: Figure size tuple
        alpha: Transparency of points
        size_scale: Scaling factor for bubble sizes (only for 'bubble' scheme)
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Assign group index to each row based on z_col value
    z_values = all_metrics_df[z_col]
    group_idx = pd.Series(index=all_metrics_df.index, dtype=int)
    
    for i, group_vals in enumerate(group_of_z_col):
        mask = z_values.isin(group_vals)
        group_idx[mask] = i
    
    # Handle values not in any group
    unassigned = group_idx.isna()
    if unassigned.any():
        group_idx[unassigned] = len(group_of_z_col)
    
    n_groups = len(group_of_z_col)
    
    # Generate labels
    if group_labels is None:
        group_labels = [f'Group {i+1}' for i in range(n_groups)]
    if unassigned.any():
        group_labels.append('Other')
    
    x = all_metrics_df[x_col]
    y = all_metrics_df[y_col]
    
    if scheme == 'color':
        # Use distinct colors for each group
        colors = plt.cm.tab10(range(len(group_labels)))
        
        for i, label in enumerate(group_labels):
            mask = group_idx == i
            if mask.any():
                ax.scatter(x[mask], y[mask], c=[colors[i]], label=label,
                           alpha=alpha, edgecolors='k', linewidth=0.5, s=60)
        
        ax.legend(title=z_col, loc='best')
        
    elif scheme == 'bubble':
        # Use different sizes for each group
        base_sizes = [(i + 1) / n_groups * size_scale for i in range(n_groups)]
        if unassigned.any():
            base_sizes.append(size_scale * 0.3)
        
        colors = plt.cm.tab10(range(len(group_labels)))
        
        for i, label in enumerate(group_labels):
            mask = group_idx == i
            if mask.any():
                ax.scatter(x[mask], y[mask], s=base_sizes[i], c=[colors[i]], 
                           label=label, alpha=alpha, edgecolors='k', linewidth=0.5)
        
        ax.legend(title=z_col, loc='best')
    
    else:
        raise ValueError(f"Unknown scheme '{scheme}'. Use 'color' or 'bubble'.")
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def basic_scatter_plot(all_metrics_df: pd.DataFrame,
                       x_col: str,
                       y_col: str,
                       title: str,
                       penalty_value: float = None,
                       figsize: tuple = (8, 6),
                       alpha: float = 0.7):
    """
    Create a basic scatter plot.
    
    Args:
        all_metrics_df: DataFrame containing the data
        x_col: Column name for x-axis
        y_col: Column name for y-axis
        penalty_value: Specific penalty value to filter data
        title: Plot title
        figsize: Figure size tuple
        alpha: Transparency of points
    """
    if penalty_value is None:
        # If no penalty value provided, plot a scatter plot for each penalty value,
        # and group all subplots together.
        unique_penalties = all_metrics_df['penalty'].unique()
        n_penalties = len(unique_penalties)
        # Show 3 subplots for each row
        n_cols = 3
        n_rows = (n_penalties + n_cols - 1) // n_cols
        fig, axs = plt.subplots(n_rows, n_cols, figsize=(figsize[0] * n_cols, figsize[1] * n_rows), sharey=True)
        axs = axs.flatten()
        for ax, pen in zip(axs, unique_penalties):
            filtered_df = all_metrics_df[all_metrics_df['penalty'] == pen]
            x = filtered_df[x_col]
            y = filtered_df[y_col]
            
            ax.scatter(x, y, c='skyblue', alpha=alpha, edgecolors='k', linewidth=0.5, s=60)
            ax.set_xlabel(x_col)
            ax.set_title(f"{title} (penalty={pen})")
            ax.grid(True, alpha=0.3)
        axs[0].set_ylabel(y_col)
        plt.tight_layout()
        plt.show()
        return

    fig, ax = plt.subplots(figsize=figsize)
    
    filtered_df = all_metrics_df[all_metrics_df['penalty'] == penalty_value]
    x = filtered_df[x_col]
    y = filtered_df[y_col]
    
    ax.scatter(x, y, c='skyblue', alpha=alpha, edgecolors='k', linewidth=0.5, s=60)
    
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
agg_metrics_center_random_df = agg_random_df_across_instances(all_metrics_center_random_df)
agg_metrics_outside_random_df = agg_random_df_across_instances(all_metrics_outside_random_df)

### collaboration rate

In [ ]:
boxplot_metric(all_metrics_center_random_df,
               metric='collaboration_rate',
               title='Collaboration Rate vs Penalty (Center Depot, Random Receivers)',  
               compare_iter0=False)
boxplot_metric(all_metrics_outside_random_df,
               metric='collaboration_rate',
                title='Collaboration Rate vs Penalty (Outside Depot, Random Receivers)',
                compare_iter0=False)


### VKT

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='VKT_km', 
               title='Total Freight VKT (Center Depot, Random Receivers)', 
               compare_iter0=True)
boxplot_metric(all_metrics_outside_random_df,
                metric='VKT_km', 
                title='Total Freight VKT (Outside Depot, Random Receivers)', 
                compare_iter0=True)

### VTT (travel time)

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='VTT_seconds', 
               title='Total Freight VTT (Center Depot, Random Receivers)', 
               compare_iter0=True)
boxplot_metric(all_metrics_outside_random_df,
                metric='VTT_seconds', 
                title='Total Freight VTT (Outside Depot, Random Receivers)', 
                compare_iter0=True)

### Ton-km travelled (tkt)

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='TKT_tonkm', 
               title='Total Freight TKT (Center Depot, Random Receivers)', 
               compare_iter0=True)
boxplot_metric(all_metrics_outside_random_df,
                metric='TKT_tonkm', 
                title='Total Freight TKT (Outside Depot, Random Receivers)', 
                compare_iter0=True)

### Fleet size

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='fleet_size', 
               title='Total Freight Fleet Size (Center Depot, Random Receivers)', 
               compare_iter0=True)
boxplot_metric(all_metrics_outside_random_df,
                metric='fleet_size', 
                title='Total Freight Fleet Size (Outside Depot, Random Receivers)', 
                compare_iter0=True)

### Carrier score

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='final_carrier_score', 
               title='Carrier Score (Center Depot, Random Receivers)', 
               iter0_metric='iter0_carrier_score')
boxplot_metric(all_metrics_outside_random_df,
                metric='final_carrier_score', 
                title='Carrier Score (Outside Depot, Random Receivers)', 
                iter0_metric='iter0_carrier_score')

### Receiver score

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='total_receiver_scores', 
               title='Total Freight Receiver Scores (Center Depot, Random Receivers)', 
               compare_iter0=False)
boxplot_metric(all_metrics_outside_random_df,
                metric='total_receiver_scores', 
                title='Total Freight Receiver Scores (Outside Depot, Random Receivers)', 
                compare_iter0=False)

### extended TWs

In [ ]:
boxplot_metric(all_metrics_center_random_df, 
               metric='total_extended_tw_hours', 
               title='Total Extended TW Hours (Center Depot, Random Receivers)', 
               compare_iter0=False)
boxplot_metric(all_metrics_outside_random_df,
                metric='total_extended_tw_hours', 
                title='Total Extended TW Hours (Outside Depot, Random Receivers)', 
                compare_iter0=False)

### Mean receiver distance to the depot
This should be set as the X , exporing the correlations with other variables (like collaboration rate, scores, etc.)

In [ ]:
scatter_3d(all_metrics_center_random_df,
           x_col='mean_receiver_dist_to_depot_euclidean_km',
           y_col='collaboration_rate',
           z_col='penalty',
           color_col=None,
           title='center-random-Mean receiver euclidean dist to depot-collab rate-penalty'
           )
scatter_color_encoded(all_metrics_center_random_df,
                      x_col='mean_receiver_dist_to_depot_euclidean_km',
                      y_col='collaboration_rate',
                      z_color_col='penalty',
                      title='center-random-Mean receiver euclidean dist to depot-collab rate-penalty'
                      )
group_scatter_plot(all_metrics_center_random_df,
                   x_col='mean_receiver_dist_to_depot_euclidean_km',
                   y_col='collaboration_rate',
                   z_col='penalty',
                   group_of_z_col=[[0], [0.0003, 0.0008, 0.0014, 0.0028], [0.0056,
                        0.0098, 0.014], [0.0167, 0.0222, 0.028]],
                    group_labels=['None', 'low', 'medium', 'high']
                   )

In [ ]:
scatter_3d(all_metrics_center_random_df,
           x_col='mean_receiver_dist_to_depot_network_km',
           y_col='collaboration_rate',
           z_col='penalty',
           color_col=None,
           title='center-random-Mean receiver nx dist to depot-collab rate-penalty'
           )
scatter_color_encoded(all_metrics_center_random_df,
                      x_col='mean_receiver_dist_to_depot_network_km',
                      y_col='collaboration_rate',
                      z_color_col='penalty',
                      title='center-random-Mean receiver nx dist to depot-collab rate-penalty'
                      )
group_scatter_plot(all_metrics_center_random_df,
                   x_col='mean_receiver_dist_to_depot_network_km',
                   y_col='collaboration_rate',
                   z_col='penalty',
                   group_of_z_col=[[0], [0.0003, 0.0008, 0.0014, 0.0028], [0.0056,
                        0.0098, 0.014], [0.0167, 0.0222, 0.028]],
                    group_labels=['None', 'low', 'medium', 'high'],
                    
                   )

In [ ]:
scatter_3d(all_metrics_outside_random_df,
           x_col='mean_receiver_dist_to_depot_network_km',
           y_col='collaboration_rate',
           z_col='penalty',
           color_col=None,
           title='outside-random-Mean receiver nx dist to depot-collab rate-penalty'
           )
scatter_color_encoded(all_metrics_outside_random_df,
                      x_col='mean_receiver_dist_to_depot_network_km',
                      y_col='collaboration_rate',
                      z_color_col='penalty',
                      title='outside-random-Mean receiver nx dist to depot-collab rate-penalty'
                      )
group_scatter_plot(all_metrics_outside_random_df,
                   x_col='mean_receiver_dist_to_depot_network_km',
                   y_col='collaboration_rate',
                   z_col='penalty',
                   group_of_z_col=[[0], [0.0003, 0.0008, 0.0014, 0.0028], [0.0056,
                        0.0098, 0.014], [0.0167, 0.0222, 0.028]],
                    group_labels=['None', 'low', 'medium', 'high'],
                    title='outside-random-Mean receiver nx dist to depot-collab rate-penalty'
                   )
basic_scatter_plot(all_metrics_outside_random_df,
                     x_col='mean_receiver_dist_to_depot_network_km',
                     y_col='collaboration_rate',
                    #  penalty_value=0.0056,
                     title='outside-random-Mean receiver nx dist to depot-collab rate at penalty=0.0056'
                     )

## Analyse spatial scenarios

In [ ]:
all_metrics_center_dispersed_df.groupby(['instance'])['collaboration_rate'].agg('mean')

In [ ]:
all_metrics_center_clustered_df.groupby(['instance'])['collaboration_rate'].agg('mean')

### collaboration rate

In [ ]:
boxplot_metric(all_metrics_center_clustered_df,
                metric='collaboration_rate',
                title='Collaboration Rate vs Penalty (Center Depot, Clustered Receivers)',  
                compare_iter0=False)
boxplot_metric(all_metrics_center_dispersed_df,
                metric='collaboration_rate',
                title='Collaboration Rate vs Penalty (Center Depot, Dispersed Receivers)',  
                compare_iter0=False)
boxplot_metric(all_metrics_outside_clustered_df,
                metric='collaboration_rate',
                title='Collaboration Rate vs Penalty (Outside Depot, Clustered Receivers)',  
                compare_iter0=False)
boxplot_metric(all_metrics_outside_dispersed_df,
                metric='collaboration_rate',
                title='Collaboration Rate vs Penalty (Outside Depot, Dispersed Receivers)',  
                compare_iter0=False)